# SaaS 30일 이탈 EDA

이 notebook은 `03_saas_feature_inventory_and_filtering.ipynb` 에서 만든
`saas_leave_next_30d_eda_ready` 데이터를 읽고,
현재 상태를 보면 앞으로 30일 안에 고객이 떠나는지에 대한 EDA를 수행한다.

## 여기서 보고 싶은 것

1. `monthly` 와 `annual` 중 누가 더 많이 떠나는가
2. `plan tier` 에 따라 이탈이 다른가
3. `days_to_expiry`, `renewal_failure`, `usage_drop` 같은 현재 상태가 실제 이탈과 연결되는가

즉 이 notebook은 `현재 상태 -> 30일 내 이탈 여부` 를 보는 실제 분석 notebook이다.

## 진행 순서

- Part 1. EDA 데이터 읽기
- Part 2. 전체 이탈 비율과 기본 분포 보기
- Part 3. monthly vs annual 비교 + 숫자형 feature별 시각화
- Part 4. plan tier 비교 + 숫자형 feature별 시각화
- Part 5. leave 여부별 숫자형 feature 시각화
- Part 6. leave_reason 분해 보기
- Part 7. 전체 숫자형 feature 분포 보기


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ModuleNotFoundError:
    plt = None
    HAS_MPL = False

try:
    import seaborn as sns
    HAS_SEABORN = True
except ModuleNotFoundError:
    sns = None
    HAS_SEABORN = False

CWD = Path.cwd().resolve()
WORKSPACE_ROOT = next(
    (path for path in [CWD, *CWD.parents] if (path / 'back_research').exists() and (path / 'docs').exists()),
    CWD,
)
NOTEBOOK_ROOT = WORKSPACE_ROOT / 'back_research' / 'wonbeenlee' / 'notebooks' / 'branch_workspaces' / 'membership_subscription_lifecycle'
EDA_READY_DIR = NOTEBOOK_ROOT / 'eda' / 'outputs' / 'eda_ready'
EDA_READY_CSV = EDA_READY_DIR / 'saas_leave_next_30d_eda_ready.csv'
EDA_READY_PARQUET = EDA_READY_DIR / 'saas_leave_next_30d_eda_ready.parquet'

NUMERIC_FEATURES = [
    'current_mrr_amount',
    'current_seats',
    'days_since_join',
    'days_to_expiry',
    'cancel_flag_30d',
    'renewal_failure_count_90d',
    'plan_change_up_flag_90d',
    'plan_change_down_flag_90d',
    'usage_intensity_30d',
    'usage_drop_ratio_30d_vs_prev_30d',
]

print({'eda_ready_csv': str(EDA_READY_CSV), 'exists': EDA_READY_CSV.exists()})
print({'matplotlib': HAS_MPL, 'seaborn': HAS_SEABORN})

## Part 1. EDA 데이터 읽기

이 notebook은 `saas_leave_next_30d_eda_ready` 한 파일만 읽는다.
이미 필요한 컬럼만 남긴 상태이므로, 여기서는 바로 분석으로 들어간다.

In [ ]:
def load_prefer_csv(csv_path: Path, parquet_path: Path) -> pd.DataFrame:
    if csv_path.exists():
        return pd.read_csv(csv_path, low_memory=False)
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    raise FileNotFoundError(f'Could not find EDA-ready export: {csv_path.name} or {parquet_path.name}. Run notebook 03 first.')

eda = load_prefer_csv(EDA_READY_CSV, EDA_READY_PARQUET)
print({'eda_shape': eda.shape})
eda.head()

In [ ]:
def plot_numeric_by_group(frame: pd.DataFrame, group_col: str, features: list[str], title_prefix: str) -> None:
    if not HAS_MPL:
        print('matplotlib is not available; skip charts.')
        return

    ncols = 2
    nrows = int(np.ceil(len(features) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, feature in zip(axes, features):
        data = frame[[group_col, feature]].dropna().copy()
        if data.empty:
            ax.set_visible(False)
            continue
        if HAS_SEABORN:
            sns.boxplot(data=data, x=group_col, y=feature, ax=ax)
        else:
            data.boxplot(column=feature, by=group_col, ax=ax)
        ax.set_title(f'{title_prefix}: {feature}')
        ax.tick_params(axis='x', rotation=20)

    for ax in axes[len(features):]:
        ax.set_visible(False)

    plt.suptitle('')
    plt.tight_layout()
    plt.show()

def plot_numeric_distributions(frame: pd.DataFrame, features: list[str], title_prefix: str) -> None:
    if not HAS_MPL:
        print('matplotlib is not available; skip charts.')
        return

    ncols = 2
    nrows = int(np.ceil(len(features) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, feature in zip(axes, features):
        data = pd.to_numeric(frame[feature], errors='coerce').dropna()
        if data.empty:
            ax.set_visible(False)
            continue
        if HAS_SEABORN:
            sns.histplot(data, bins=30, ax=ax)
        else:
            data.plot(kind='hist', bins=30, ax=ax)
        ax.set_title(f'{title_prefix}: {feature}')

    for ax in axes[len(features):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()

## Part 2. 전체 이탈 비율과 기본 분포 보기

먼저 전체 leave 비율과 기본 segment 분포를 본다.
이 단계에서 데이터가 극단적으로 치우쳐 있는지부터 확인한다.

In [ ]:
overview = pd.DataFrame([
    {
        'row_count': len(eda),
        'leave_rate': round(float(eda['leave_next_30d'].mean()), 4),
        'min_as_of_date': eda['as_of_date'].min(),
        'max_as_of_date': eda['as_of_date'].max(),
        'billing_values': ', '.join(sorted(eda['current_billing_frequency'].dropna().astype(str).unique().tolist())),
        'plan_values': ', '.join(sorted(eda['current_plan_tier'].dropna().astype(str).unique().tolist())),
    }
])
overview

In [ ]:
segment_counts = pd.concat([
    eda['current_billing_frequency'].value_counts(dropna=False).rename('current_billing_frequency'),
    eda['current_plan_tier'].value_counts(dropna=False).rename('current_plan_tier'),
    eda['current_auto_renew_flag'].value_counts(dropna=False).rename('current_auto_renew_flag'),
    eda['leave_reason'].value_counts(dropna=False).rename('leave_reason'),
], axis=1)
segment_counts

## Part 3. monthly vs annual 비교

질문:
- monthly 고객이 annual 고객보다 더 잘 빠져나가는가?
- 숫자형 feature 수준도 같이 다른가?

In [ ]:
billing_summary = eda.groupby('current_billing_frequency').agg(
    rows=('as_of_date', 'size'),
    leave_rate=('leave_next_30d', 'mean'),
    avg_days_to_expiry=('days_to_expiry', 'mean'),
    avg_renewal_failure_90d=('renewal_failure_count_90d', 'mean'),
    avg_usage_30d=('usage_intensity_30d', 'mean'),
    avg_usage_drop=('usage_drop_ratio_30d_vs_prev_30d', 'mean'),
)
billing_summary.round(4)

### monthly vs annual 기준 숫자형 feature 시각화

아래 차트는 각 숫자형 feature가 `current_billing_frequency` 별로 어떻게 분포하는지 보여준다.

In [ ]:
plot_numeric_by_group(eda, 'current_billing_frequency', NUMERIC_FEATURES, 'billing_frequency')

## Part 4. plan tier 비교

질문:
- Basic / Pro / Enterprise 에 따라 이탈률이 다른가?
- 숫자형 feature 수준도 같이 다른가?

In [ ]:
plan_summary = eda.groupby('current_plan_tier').agg(
    rows=('as_of_date', 'size'),
    leave_rate=('leave_next_30d', 'mean'),
    avg_days_to_expiry=('days_to_expiry', 'mean'),
    avg_renewal_failure_90d=('renewal_failure_count_90d', 'mean'),
    avg_usage_30d=('usage_intensity_30d', 'mean'),
    avg_usage_drop=('usage_drop_ratio_30d_vs_prev_30d', 'mean'),
)
plan_summary.round(4)

### plan tier 기준 숫자형 feature 시각화

아래 차트는 각 숫자형 feature가 `current_plan_tier` 별로 어떻게 분포하는지 보여준다.

In [ ]:
plot_numeric_by_group(eda, 'current_plan_tier', NUMERIC_FEATURES, 'plan_tier')

## Part 5. 현재 상태 feature와 이탈 관계 보기

질문:
- 어떤 현재 상태가 실제 leave 와 같이 움직이는가?

여기서는 숫자형 feature를 `leave_next_30d` 기준으로 나눠서 본다.

In [ ]:
feature_by_leave = eda.groupby('leave_next_30d').agg(
    avg_days_since_join=('days_since_join', 'mean'),
    avg_days_to_expiry=('days_to_expiry', 'mean'),
    avg_cancel_flag_30d=('cancel_flag_30d', 'mean'),
    avg_renewal_failure_90d=('renewal_failure_count_90d', 'mean'),
    avg_plan_change_up=('plan_change_up_flag_90d', 'mean'),
    avg_plan_change_down=('plan_change_down_flag_90d', 'mean'),
    avg_usage_30d=('usage_intensity_30d', 'mean'),
    avg_usage_drop=('usage_drop_ratio_30d_vs_prev_30d', 'mean'),
)
feature_by_leave.round(4)

### leave 여부 기준 숫자형 feature 시각화

아래 차트는 각 숫자형 feature가 `leave_next_30d` 기준으로 어떻게 달라지는지 보여준다.

In [ ]:
plot_numeric_by_group(eda, 'leave_next_30d', NUMERIC_FEATURES, 'leave_next_30d')

## Part 6. leave_reason 분해 보기

질문:
- leave 한 고객 중에서 직접 churn event로 잡히는 경우가 많은가
- 아니면 non-renewal 쪽이 많은가


In [ ]:
leave_reason_summary = eda.groupby('leave_reason').agg(
    rows=('as_of_date', 'size'),
    avg_days_to_expiry=('days_to_expiry', 'mean'),
    avg_usage_drop=('usage_drop_ratio_30d_vs_prev_30d', 'mean'),
    avg_renewal_failure_90d=('renewal_failure_count_90d', 'mean'),
)
leave_reason_summary.round(4)

## Part 7. 전체 숫자형 feature 분포 보기

마지막으로 숫자형 feature 자체의 전체 분포를 하나씩 본다.
이 단계는 값 범위와 이상치 감을 잡기 위한 단계다.

In [ ]:
plot_numeric_distributions(eda, NUMERIC_FEATURES, 'overall_distribution')

## Part 8. 결론

현재 생성된 `saas_leave_next_30d_eda_ready` 기준 핵심 결론은 아래와 같다.

- 전체 row는 `5,807` 개이고, `leave_next_30d` 비율은 약 `9.21%` 다. 즉 라벨이 완전히 희소한 수준은 아니어서 1차 EDA와 baseline 모델링은 가능한 수준이다.
- `monthly` 와 `annual` 의 차이는 있긴 하지만 아주 크지는 않다. 현재 기준으로 `annual` leave 비율은 약 `9.60%`, `monthly` 는 약 `8.67%` 였다. 다만 `annual` 쪽이 평균 `days_to_expiry` 가 더 짧고 renewal failure도 조금 더 높아서, expiry/renewal 압력은 annual에서 더 강하게 잡힌다.
- plan tier별로는 단순히 tier가 높을수록 덜 떠난다고 보기는 어렵다. 현재 기준 leave 비율은 `Pro 8.23%`, `Basic 9.36%`, `Enterprise 9.71%` 로, 오히려 `Pro` 가 가장 낮고 `Enterprise` 가 가장 높게 나왔다. 즉 tier 하나만으로 monotonic한 해석을 하면 안 된다.
- leave한 row와 stay row를 비교하면 가장 분명한 차이는 `days_to_expiry` 다. leave row의 평균 `days_to_expiry` 는 약 `49.67` 일, stay row는 약 `151.01` 일이었다. 즉 만료가 가까운 상태가 실제 이탈과 가장 강하게 연결된다.
- `days_since_join` 도 차이가 있다. leave row의 평균은 약 `188.64` 일, stay row는 약 `245.22` 일이었다. 즉 비교적 tenure가 짧은 고객이 더 쉽게 빠져나가는 경향이 보인다.
- `cancel_flag_30d` 와 `plan_change_down_flag_90d` 는 leave row에서 조금 더 높다. 따라서 최근 cancel 신호와 다운그레이드 경험은 이탈 전조로 볼 만하다.
- 반면 `usage_intensity_30d` 와 `usage_drop_ratio_30d_vs_prev_30d` 는 leave/stay 평균 차이가 생각보다 크지 않았다. 즉 현재 전처리 기준으로는 usage 계열이 leave를 강하게 분리하지 못하고 있다. usage window 정의를 다시 보거나, usage 관련 컬럼을 더 정교하게 만들어볼 필요가 있다.
- leave reason을 보면 실제 leave의 대부분은 `churn_event` 경로였고, `non_renewal` 은 소수였다. 다만 `non_renewal` row에서는 평균 `days_to_expiry` 가 약 `11.52` 일, 평균 `renewal_failure_count_90d` 가 `0.3333` 으로 높게 나와 expiry/renewal 신호가 훨씬 직접적으로 작동한다.

### 지금 단계에서 유지할 핵심 feature

- `days_to_expiry`
- `days_since_join`
- `cancel_flag_30d`
- `plan_change_down_flag_90d`
- `renewal_failure_count_90d`

### 다음에 다시 봐야 할 것

- `usage_intensity_30d`
- `usage_drop_ratio_30d_vs_prev_30d`

이 둘은 지금 기준으로는 leave와의 차이가 약해서, window 정의나 집계 방식 자체를 다시 보는 게 맞다.

### 실무적으로 바로 이어질 다음 작업

1. `days_to_expiry`, `renewal_failure_count_90d`, `days_since_join` 중심 baseline 모델을 먼저 돌린다.
2. usage feature는 현재 정의를 그대로 믿지 말고 다시 설계 후보를 만든다.
3. `monthly` / `annual` 을 같은 `30일 horizon` 으로 보는 것이 맞는지도 다시 검토한다.